In [ ]:
!pip install gradio scikit-learn pandas -q

import re
import gradio as gr
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

APP_NAME = "TRỢ LÝ ẢO HỖ TRỢ TRẢ LỜI CÁC CÂU HỎI THƯỜNG GẶP VỀ SỨC KHỎE GIỚI TÍNH"

def kb(intent, topic, keywords, answer, advice, seek_help):
    return {
        "intent": intent,
        "topic": topic,
        "keywords": keywords,
        "answer": answer,
        "advice": advice,
        "seek_help": seek_help
    }

knowledge_base = [
    kb("puberty", "dấu hiệu dậy thì ở nam",
       ["dấu hiệu dậy thì ở nam", "nam dậy thì", "con trai dậy thì"],
       "Dậy thì ở nam thường có các thay đổi như cao nhanh, mọc lông, giọng trầm hơn, cơ thể có mùi hơn, da dễ nổi mụn, cơ quan sinh dục phát triển và có thể xuất hiện mộng tinh.",
       "Bạn nên giữ vệ sinh cá nhân, ngủ đủ, ăn uống đều và không so sánh cơ thể mình với bạn bè vì mỗi người phát triển khác nhau.",
       "Nên hỏi bác sĩ nếu trên khoảng 14-15 tuổi vẫn chưa có dấu hiệu dậy thì rõ hoặc có đau, sưng, bất thường kéo dài."),

    kb("puberty", "dậy thì ở nữ bắt đầu khi nào",
       ["dậy thì ở nữ bắt đầu khi nào", "nữ dậy thì", "con gái dậy thì"],
       "Dậy thì ở nữ thường bắt đầu khoảng 8-13 tuổi. Có thể có phát triển ngực, mọc lông, thay đổi vóc dáng, da dễ nổi mụn và xuất hiện kinh nguyệt.",
       "Hãy theo dõi cơ thể, giữ vệ sinh vùng kín đúng cách và chuẩn bị kiến thức về kinh nguyệt.",
       "Nên đi khám nếu dậy thì quá sớm, quá muộn, đau nhiều hoặc ra máu bất thường."),

    kb("puberty", "vì sao tuổi dậy thì hay nổi mụn",
       ["vì sao tuổi dậy thì hay nổi mụn", "dậy thì nổi mụn", "mụn tuổi dậy thì"],
       "Tuổi dậy thì dễ nổi mụn do thay đổi hormone làm tuyến dầu hoạt động mạnh hơn, lỗ chân lông dễ bít tắc hơn.",
       "Rửa mặt nhẹ nhàng, không nặn mụn, ngủ đủ và hạn chế bôi nhiều sản phẩm không rõ nguồn.",
       "Nên khám da liễu nếu mụn viêm nặng, đau nhiều, để lại sẹo hoặc kéo dài."),

    kb("puberty", "giọng nói thay đổi ở tuổi dậy thì",
       ["giọng nói thay đổi", "vỡ giọng", "giọng trầm", "giọng đổi"],
       "Giọng nói thay đổi ở tuổi dậy thì là bình thường, đặc biệt ở nam. Giọng có thể lúc cao lúc thấp trong một thời gian trước khi ổn định.",
       "Bạn không cần quá lo nếu chỉ là thay đổi giọng bình thường và không đau rát kéo dài.",
       "Nên đi khám nếu khàn tiếng kéo dài, đau họng nhiều hoặc mất giọng lâu ngày."),

    kb("puberty", "cơ thể có mùi nhiều hơn khi dậy thì",
       ["cơ thể có mùi", "mùi cơ thể dậy thì", "hôi nách dậy thì"],
       "Khi dậy thì, tuyến mồ hôi và tuyến dầu hoạt động mạnh hơn nên cơ thể có thể có mùi rõ hơn.",
       "Tắm rửa đều, thay quần áo sau khi ra mồ hôi, dùng đồ thoáng và giữ vùng nách/vùng kín sạch khô.",
       "Nên đi khám nếu mùi rất nặng, kèm ngứa, viêm da hoặc tiết dịch bất thường."),

    kb("puberty", "mộng tinh có bình thường không",
       ["mộng tinh", "mộng tinh có bình thường không", "xuất tinh khi ngủ"],
       "Mộng tinh ở tuổi dậy thì nam có thể là hiện tượng sinh lý bình thường do cơ thể đang phát triển.",
       "Bạn không cần xấu hổ. Hãy vệ sinh sạch sẽ, ngủ nghỉ điều độ và tránh lo lắng quá mức.",
       "Nên trao đổi với bác sĩ nếu đau, tiểu buốt, có máu hoặc lo lắng kéo dài."),

    kb("puberty", "kinh nguyệt đầu tiên có đáng lo không",
       ["kinh nguyệt đầu tiên", "lần đầu có kinh", "mới có kinh"],
       "Kinh nguyệt đầu tiên thường là dấu hiệu dậy thì ở nữ. Những kỳ đầu có thể chưa đều, lượng máu có thể ít hoặc màu nâu đỏ.",
       "Bạn nên dùng băng vệ sinh phù hợp, thay thường xuyên, vệ sinh nhẹ nhàng và ghi lại ngày có kinh.",
       "Nên đi khám nếu máu ra quá nhiều, đau dữ dội, choáng hoặc ra máu kéo dài bất thường."),

    kb("puberty", "tuổi dậy thì kéo dài bao lâu",
       ["tuổi dậy thì kéo dài bao lâu", "dậy thì kéo dài", "bao lâu hết dậy thì"],
       "Dậy thì thường kéo dài vài năm. Tốc độ thay đổi ở mỗi người khác nhau nên không có một mốc giống hệt cho tất cả.",
       "Hãy chú ý ăn ngủ, vận động, vệ sinh và tâm lý thay vì quá lo vì phát triển khác bạn bè.",
       "Nên đi khám nếu có dấu hiệu dậy thì quá sớm, quá muộn hoặc bất thường rõ."),

    kb("menstruation", "trễ kinh bao lâu thì cần lo lắng",
       ["trễ kinh bao lâu", "chậm kinh bao lâu", "trễ kinh cần lo"],
       "Trễ kinh vài ngày có thể do stress, thức khuya, thay đổi sinh hoạt hoặc cân nặng. Nếu có quan hệ nguy cơ thì cần nghĩ đến khả năng mang thai.",
       "Theo dõi thêm vài ngày, ngủ đủ, giảm stress. Nếu có quan hệ nguy cơ, nên thử thai đúng thời điểm.",
       "Nên đi khám nếu trễ kinh kéo dài nhiều tuần, lặp lại nhiều chu kỳ hoặc kèm đau/ra máu bất thường."),

    kb("menstruation", "kinh nguyệt không đều",
       ["kinh nguyệt không đều", "kinh không đều", "rối loạn kinh nguyệt"],
       "Kinh nguyệt không đều có thể gặp ở tuổi dậy thì, khi stress, thức khuya, thay đổi cân nặng hoặc nội tiết chưa ổn định.",
       "Ghi lại chu kỳ 2-3 tháng, ngủ đủ, ăn uống đều và giảm căng thẳng.",
       "Nên đi khám nếu mất kinh nhiều tháng, rong kinh, đau dữ dội hoặc ra máu bất thường."),

    kb("menstruation", "đau bụng kinh nhiều",
       ["đau bụng kinh nhiều", "đau bụng kinh", "đau ngày đèn đỏ"],
       "Đau bụng kinh nhẹ đến vừa có thể gặp do tử cung co bóp. Đau quá nhiều hoặc ảnh hưởng sinh hoạt thì cần chú ý.",
       "Có thể nghỉ ngơi, chườm ấm bụng dưới, uống đủ nước và tránh thức khuya.",
       "Nên đi khám nếu đau dữ dội, ngất, sốt, ra máu quá nhiều hoặc đau khác thường."),

    kb("menstruation", "vì sao kinh nguyệt đến sớm hoặc muộn",
       ["kinh nguyệt đến sớm", "kinh nguyệt đến muộn", "kinh tới sớm", "kinh tới muộn", "chu kỳ thay đổi"],
       "Kinh nguyệt có thể đến sớm hoặc muộn do stress, thức khuya, thay đổi cân nặng, sinh hoạt thất thường, vận động quá sức hoặc thay đổi nội tiết.",
       "Theo dõi chu kỳ vài tháng, ngủ đủ, ăn uống đều và giảm căng thẳng.",
       "Nên đi khám nếu rối loạn kéo dài, mất kinh nhiều tháng, đau dữ dội hoặc ra máu bất thường."),

    kb("menstruation", "có nên tắm khi đang có kinh",
       ["có nên tắm khi đang có kinh", "đang có kinh có tắm được không", "tắm khi tới tháng"],
       "Có thể tắm khi đang có kinh. Vệ sinh đúng cách còn giúp cơ thể sạch sẽ và dễ chịu hơn.",
       "Tắm bằng nước sạch, thay băng vệ sinh đều, giữ vùng kín khô thoáng và tránh thụt rửa sâu.",
       "Nên đi khám nếu có mùi hôi, ngứa rát, đau nhiều hoặc ra máu bất thường."),

    kb("menstruation", "máu kinh màu nâu",
       ["máu kinh màu nâu", "kinh màu nâu", "máu nâu khi có kinh"],
       "Máu kinh màu nâu có thể gặp ở đầu hoặc cuối kỳ kinh do máu ra chậm hơn. Nếu không đau nhiều, không mùi hôi, không kéo dài thì thường chưa quá đáng lo.",
       "Theo dõi lượng máu, số ngày hành kinh và triệu chứng kèm theo.",
       "Nên đi khám nếu máu nâu kéo dài, có mùi hôi, đau nhiều, ra máu giữa kỳ hoặc nghi ngờ mang thai."),

    kb("menstruation", "căng thẳng có làm trễ kinh không",
       ["căng thẳng có làm trễ kinh", "stress làm trễ kinh", "lo lắng làm chậm kinh"],
       "Căng thẳng, lo lắng và thức khuya có thể ảnh hưởng hormone, từ đó làm chu kỳ kinh đến sớm, muộn hoặc không đều.",
       "Ngủ đủ, giảm stress, ăn uống đều và theo dõi chu kỳ.",
       "Nên đi khám nếu trễ kinh kéo dài hoặc có quan hệ nguy cơ mà que thử không rõ kết quả."),

    kb("menstruation", "khi nào đi khám rối loạn kinh nguyệt",
       ["khi nào đi khám rối loạn kinh nguyệt", "rối loạn kinh nguyệt đi khám", "kinh nguyệt bất thường"],
       "Rối loạn kinh nguyệt cần chú ý hơn nếu kéo dài, lặp lại nhiều chu kỳ hoặc kèm triệu chứng bất thường.",
       "Ghi lại ngày kinh, lượng máu, số ngày ra máu, đau bụng và các thay đổi sinh hoạt.",
       "Nên đi khám nếu mất kinh nhiều tháng, rong kinh, đau dữ dội, ra máu giữa kỳ hoặc chóng mặt/mệt lả."),

    kb("contraception", "bao cao su có hiệu quả tránh thai không",
       ["bao cao su có hiệu quả tránh thai", "bao cao su tránh thai", "dùng bao có an toàn không"],
       "Bao cao su giúp giảm nguy cơ mang thai ngoài ý muốn và giảm nguy cơ một số bệnh lây truyền qua đường tình dục nếu dùng đúng và nhất quán.",
       "Cần dùng từ đầu đến cuối theo hướng dẫn, kiểm tra bao còn hạn và không dùng lại.",
       "Nên hỏi nhân viên y tế nếu bao rách, tuột hoặc có tình huống nguy cơ."),

    kb("pregnancy_risk", "quan hệ lần đầu có thể mang thai không",
       ["quan hệ lần đầu có thai không", "lần đầu có mang thai không", "quan hệ lần đầu"],
       "Quan hệ lần đầu vẫn có thể mang thai nếu có tinh dịch tiếp xúc với vùng âm đạo, dù đây là lần đầu.",
       "Nếu chưa muốn mang thai, cần dùng biện pháp tránh thai phù hợp. Nếu đã có tình huống nguy cơ, theo dõi chu kỳ và thử thai đúng thời điểm.",
       "Nên hỏi dược sĩ/nhân viên y tế nếu vừa có nguy cơ và chưa biết xử trí."),

    kb("pregnancy_risk", "cọ xát bên ngoài có mang thai không",
       ["cọ xát bên ngoài", "cọ xát ngoài có thai", "cọ xát có mang thai"],
       "Chỉ cọ xát bên ngoài, đặc biệt khi còn mặc quần áo, thường có nguy cơ mang thai rất thấp. Nguy cơ tăng nếu tinh dịch tiếp xúc trực tiếp gần vùng âm đạo.",
       "Bình tĩnh xác định có tiếp xúc trực tiếp hay không và theo dõi kỳ kinh.",
       "Nên hỏi nhân viên y tế nếu có xuất tinh gần vùng kín, trễ kinh hoặc không chắc tình huống."),

    kb("contraception", "thuốc tránh thai khẩn cấp dùng khi nào",
       ["thuốc tránh thai khẩn cấp dùng khi nào", "khi nào uống thuốc tránh thai khẩn cấp"],
       "Thuốc tránh thai khẩn cấp là biện pháp dự phòng sau tình huống có nguy cơ mang thai ngoài ý muốn, không phải biện pháp dùng thường xuyên.",
       "Chỉ cân nhắc khi có nguy cơ rõ và nên hỏi dược sĩ/nhân viên y tế để dùng đúng.",
       "Nên đi khám/hỏi dược sĩ nếu nôn nhiều, rối loạn kinh kéo dài hoặc không chắc cách dùng."),

    kb("contraception", "dùng thuốc tránh thai khẩn cấp nhiều có hại không",
       ["dùng thuốc tránh thai khẩn cấp nhiều", "uống thuốc khẩn cấp nhiều", "thuốc tránh thai khẩn cấp có hại"],
       "Lạm dụng thuốc tránh thai khẩn cấp có thể gây rối loạn kinh nguyệt, buồn nôn, mệt mỏi hoặc ảnh hưởng nội tiết tạm thời.",
       "Không nên dùng như biện pháp thường xuyên. Nên tìm biện pháp tránh thai ổn định hơn nếu cần.",
       "Nên hỏi nhân viên y tế nếu rối loạn kinh kéo dài hoặc dùng nhiều lần."),

    kb("contraception", "bao cao su bị rách",
       ["bao cao su bị rách", "rách bao", "tuột bao"],
       "Bao cao su bị rách hoặc tuột làm tăng nguy cơ mang thai ngoài ý muốn và bệnh lây truyền qua đường tình dục.",
       "Cần xác định có xuất tinh không, thời điểm xảy ra và có tiếp xúc trực tiếp không.",
       "Nên hỏi dược sĩ/nhân viên y tế càng sớm càng tốt nếu vừa có tình huống nguy cơ."),

    kb("contraception", "cần dùng bao cao su khi chưa xuất tinh không",
       ["chưa xuất tinh có cần dùng bao", "chưa xuất tinh có thai không", "có cần dùng bao cao su khi chưa xuất tinh"],
       "Vẫn nên dùng bao cao su từ đầu vì việc kiểm soát thời điểm xuất tinh không luôn chắc chắn và bao cao su còn giúp giảm nguy cơ STI.",
       "Không nên chỉ dựa vào xuất tinh ngoài để tránh thai.",
       "Nên tư vấn nhân viên y tế nếu đã có tình huống nguy cơ và đang lo lắng."),

    kb("contraception", "sử dụng bao cao su đúng cách",
       ["sử dụng bao cao su đúng cách", "dùng bao cao su đúng cách", "làm sao dùng bao cao su"],
       "Bao cao su cần dùng đúng theo hướng dẫn trên bao bì, đúng loại, còn hạn, không rách và không tái sử dụng.",
       "Bảo quản nơi khô mát, kiểm tra hạn dùng, dùng nhất quán và không dùng chung với sản phẩm có thể làm hỏng bao.",
       "Nếu bao rách/tuột hoặc dùng sai, nên hỏi dược sĩ/nhân viên y tế về xử trí nguy cơ."),

    kb("gynecology", "vì sao vùng kín bị ngứa",
       ["vùng kín bị ngứa", "ngứa vùng kín", "vì sao vùng kín ngứa"],
       "Ngứa vùng kín có thể do kích ứng, vệ sinh chưa phù hợp, mặc đồ quá chật, nấm/viêm nhiễm hoặc STI nếu có yếu tố nguy cơ.",
       "Giữ vùng kín khô thoáng, tránh gãi, tránh thụt rửa sâu và không tự dùng thuốc mạnh.",
       "Nên đi khám nếu ngứa kéo dài, kèm khí hư mùi hôi, đau rát, tiểu buốt hoặc có quan hệ nguy cơ."),

    kb("gynecology", "ra khí hư nhiều",
       ["ra khí hư nhiều", "khí hư nhiều", "dịch âm đạo nhiều"],
       "Khí hư có thể tăng theo chu kỳ kinh, khi rụng trứng hoặc do thay đổi hormone. Nếu không mùi, không ngứa rát, có thể là sinh lý.",
       "Theo dõi màu, mùi, độ đặc và triệu chứng kèm theo.",
       "Nên đi khám nếu khí hư vàng/xanh/xám, vón cục, mùi hôi hoặc kèm ngứa rát."),

    kb("gynecology", "khí hư có mùi lạ",
       ["khí hư có mùi lạ", "khí hư mùi hôi", "dịch có mùi"],
       "Khí hư có mùi lạ có thể gợi ý mất cân bằng hoặc viêm nhiễm vùng kín.",
       "Không tự đặt thuốc hoặc dùng dung dịch mạnh. Vệ sinh nhẹ nhàng và giữ khô thoáng.",
       "Nên đi khám nếu mùi hôi kéo dài, kèm ngứa rát, đau bụng dưới, tiểu buốt hoặc dịch đổi màu."),

    kb("hygiene_body", "vệ sinh vùng kín đúng cách",
       ["vệ sinh vùng kín đúng cách", "vệ sinh vùng kín", "rửa vùng kín"],
       "Vệ sinh vùng kín đúng cách là rửa nhẹ nhàng bên ngoài, giữ khô thoáng, không thụt rửa sâu và không dùng sản phẩm quá mạnh.",
       "Thay đồ lót sạch, chọn đồ thoáng, lau khô nhẹ nhàng sau khi vệ sinh.",
       "Nên đi khám nếu có ngứa, mùi hôi, khí hư bất thường hoặc đau kéo dài."),

    kb("hygiene_body", "dùng dung dịch vệ sinh mỗi ngày",
       ["dùng dung dịch vệ sinh mỗi ngày", "dung dịch vệ sinh hằng ngày", "có nên dùng dung dịch vệ sinh"],
       "Dung dịch vệ sinh có thể dùng nếu phù hợp và không gây kích ứng, nhưng không bắt buộc phải dùng mỗi ngày với mọi người.",
       "Chọn loại dịu nhẹ, dùng bên ngoài, tránh thụt rửa sâu và ngưng dùng nếu ngứa/rát hơn.",
       "Nên đi khám nếu có kích ứng, ngứa rát, mùi hôi hoặc dịch bất thường."),

    kb("gynecology", "mặc quần quá chật gây viêm nhiễm",
       ["mặc quần quá chật", "quần chật gây viêm", "đồ lót chật"],
       "Mặc quần quá chật hoặc bí có thể làm vùng kín ẩm, nóng và dễ kích ứng hơn, từ đó tăng nguy cơ khó chịu hoặc viêm nhiễm.",
       "Nên chọn đồ lót thoáng, thay sau khi ra mồ hôi và giữ vùng kín khô.",
       "Nên đi khám nếu ngứa rát, khí hư lạ hoặc mùi hôi kéo dài."),

    kb("gynecology", "khi nào cần đi khám phụ khoa",
       ["khi nào cần đi khám phụ khoa", "đi khám phụ khoa khi nào", "cần khám phụ khoa"],
       "Nên đi khám phụ khoa khi có triệu chứng bất thường kéo dài hoặc ảnh hưởng sinh hoạt.",
       "Theo dõi kỹ triệu chứng: ngứa, rát, khí hư, mùi, đau bụng dưới, tiểu buốt, ra máu bất thường.",
       "Đi khám sớm nếu đau dữ dội, chảy máu bất thường, dịch hôi rõ, sốt hoặc nghi ngờ STI."),

    kb("gynecology", "viêm nhiễm vùng kín có tự hết không",
       ["viêm nhiễm vùng kín có tự hết", "viêm vùng kín tự hết không"],
       "Một số kích ứng nhẹ có thể cải thiện khi vệ sinh và sinh hoạt đúng hơn, nhưng viêm nhiễm thật sự không nên chủ quan.",
       "Không tự dùng thuốc khi chưa rõ nguyên nhân. Giữ khô thoáng, tránh thụt rửa sâu và theo dõi triệu chứng.",
       "Nên đi khám nếu triệu chứng kéo dài, nặng hơn, có mùi hôi, đau rát hoặc dịch bất thường."),

    kb("sti_std", "HIV lây qua những đường nào",
       ["hiv lây qua đường nào", "hiv lây như thế nào", "đường lây hiv"],
       "HIV có thể lây qua máu, quan hệ tình dục không an toàn và từ mẹ sang con. HIV không lây qua tiếp xúc thông thường như bắt tay, ăn chung hoặc dùng chung nhà vệ sinh.",
       "Phòng tránh bằng quan hệ an toàn, không dùng chung kim tiêm và xét nghiệm khi có nguy cơ.",
       "Nên xét nghiệm nếu có quan hệ nguy cơ, tiếp xúc máu nguy cơ hoặc lo lắng sau tình huống cụ thể."),

    kb("sti_std", "quan hệ không an toàn nguy hiểm gì",
       ["quan hệ không an toàn", "không dùng bao nguy hiểm", "quan hệ không bảo vệ"],
       "Quan hệ không an toàn có thể làm tăng nguy cơ mang thai ngoài ý muốn và bệnh lây truyền qua đường tình dục như HIV, HPV, lậu, giang mai, chlamydia hoặc herpes.",
       "Nên dùng bao cao su đúng cách, xét nghiệm khi có nguy cơ và trao đổi thẳng thắn về sức khỏe với bạn tình.",
       "Nên đi khám/xét nghiệm nếu có triệu chứng bất thường hoặc có tình huống nguy cơ."),

    kb("sti_std", "dấu hiệu bệnh lây truyền qua đường tình dục",
       ["dấu hiệu bệnh lây truyền", "dấu hiệu sti", "dấu hiệu std"],
       "Dấu hiệu STI có thể gồm tiểu buốt, dịch lạ, mùi hôi, đau vùng bụng dưới, nổi mụn/vết loét hoặc ngứa rát. Nhưng nhiều STI có thể ít hoặc không có triệu chứng.",
       "Không nên tự đoán bệnh qua mạng. Nếu có nguy cơ, xét nghiệm là cách đáng tin cậy hơn.",
       "Nên đi khám nếu có quan hệ nguy cơ hoặc triệu chứng bất thường."),

    kb("sti_std", "dùng bao cao su có phòng bệnh không",
       ["bao cao su phòng bệnh", "bao cao su phòng sti", "dùng bao có phòng bệnh"],
       "Bao cao su dùng đúng và nhất quán giúp giảm nguy cơ nhiều STI, gồm HIV, nhưng không bảo vệ tuyệt đối với mọi bệnh lây qua tiếp xúc da vùng không được che phủ.",
       "Dùng bao đúng cách, xét nghiệm định kỳ nếu có nguy cơ và tiêm vaccine phù hợp như HPV/viêm gan B nếu được tư vấn.",
       "Nên hỏi nhân viên y tế nếu có triệu chứng hoặc từng có quan hệ nguy cơ."),

    kb("sti_std", "khi nào nên xét nghiệm HIV",
       ["khi nào xét nghiệm hiv", "nên xét nghiệm hiv khi nào", "test hiv"],
       "Nên xét nghiệm HIV nếu có quan hệ không an toàn, rách/tuột bao, tiếp xúc máu nguy cơ hoặc lo lắng sau tình huống nguy cơ.",
       "Không nên chỉ dựa vào triệu chứng để đoán HIV. Hãy đến cơ sở y tế/tư vấn xét nghiệm để được hướng dẫn thời điểm phù hợp.",
       "Nên tìm tư vấn y tế nếu vừa có phơi nhiễm nguy cơ hoặc quá lo lắng."),

    kb("sti_std", "HPV là gì",
       ["hpv là gì", "virus hpv", "sùi mào gà"],
       "HPV là một nhóm virus lây truyền chủ yếu qua tiếp xúc tình dục. Một số type HPV có thể gây mụn cóc sinh dục, một số type liên quan đến ung thư cổ tử cung.",
       "Có thể phòng ngừa bằng vaccine HPV theo tư vấn y tế và quan hệ an toàn.",
       "Nên đi khám nếu có mụn/vết bất thường vùng sinh dục hoặc muốn tư vấn tiêm vaccine."),

    kb("sti_std", "STI có chữa được không",
       ["sti có chữa được không", "std có chữa được không", "bệnh lây có chữa được không"],
       "Một số STI do vi khuẩn có thể điều trị được bằng thuốc phù hợp; một số STI do virus có thể kiểm soát triệu chứng và giảm nguy cơ lây truyền nhưng cần theo dõi y tế.",
       "Không tự mua thuốc dùng khi chưa xét nghiệm/khám vì dễ điều trị sai.",
       "Nên đi khám chuyên khoa hoặc cơ sở xét nghiệm nếu nghi ngờ STI."),

    kb("sti_std", "phòng tránh bệnh lây truyền qua đường tình dục",
       ["phòng tránh sti", "phòng tránh std", "phòng bệnh lây truyền qua đường tình dục"],
       "Phòng tránh STI gồm dùng bao cao su đúng cách, xét nghiệm khi có nguy cơ, tiêm vaccine phù hợp và tránh quan hệ khi có tổn thương bất thường.",
       "Trao đổi trung thực với bạn tình về sức khỏe và không dùng chung vật dụng có nguy cơ dính máu.",
       "Nên tư vấn y tế nếu có nguy cơ hoặc chưa rõ cách phòng tránh."),

    kb("lgbtq", "LGBTQ là gì",
       ["lgbtq là gì", "lgbt là gì", "lgbtq+"],
       "LGBTQ+ là thuật ngữ chỉ các nhóm đa dạng về xu hướng tính dục và bản dạng giới, ví dụ đồng tính, song tính, chuyển giới và một số nhóm khác. LGBTQ+ không phải bệnh.",
       "Hãy tiếp cận chủ đề này bằng sự tôn trọng, không phán xét và không ép bản thân phải gắn nhãn ngay.",
       "Nên tìm người đáng tin cậy hoặc chuyên gia tâm lý nếu bạn bị kỳ thị, áp lực hoặc hoang mang kéo dài."),

    kb("lgbtq", "khác biệt giới tính có bình thường không",
       ["khác biệt giới tính", "cảm thấy khác biệt giới tính", "bối rối giới tính"],
       "Cảm thấy khác biệt hoặc bối rối về giới tính/xu hướng tính dục có thể xảy ra. Điều đó không có nghĩa bạn bị bệnh.",
       "Cho bản thân thời gian tìm hiểu cảm xúc, tránh tự ép mình phải trả lời ngay.",
       "Tìm hỗ trợ nếu bị áp lực, kỳ thị hoặc lo lắng kéo dài."),

    kb("lgbtq", "làm sao hiểu rõ bản thân hơn",
       ["làm sao hiểu rõ bản thân", "hiểu xu hướng tính dục", "không biết mình là gì"],
       "Hiểu bản thân là quá trình cần thời gian. Cảm xúc, sự hấp dẫn và bản dạng có thể cần quan sát lâu dài.",
       "Bạn có thể ghi lại cảm xúc, đọc nguồn tin đáng tin cậy và nói chuyện với người an toàn.",
       "Nên tìm chuyên gia tư vấn nếu hoang mang ảnh hưởng học tập, ngủ nghỉ hoặc tinh thần."),

    kb("lgbtq", "có nên chia sẻ xu hướng tính dục với gia đình",
       ["chia sẻ xu hướng tính dục", "come out với gia đình", "có nên come out"],
       "Chia sẻ xu hướng tính dục với gia đình là quyết định cá nhân. Điều quan trọng nhất là sự an toàn và sự sẵn sàng của bạn.",
       "Chọn thời điểm phù hợp, chuẩn bị tâm lý và có người đáng tin cậy hỗ trợ nếu cần.",
       "Không nên tự ép mình chia sẻ nếu môi trường chưa an toàn."),

    kb("lgbtq", "người LGBTQ cần chăm sóc sức khỏe đặc biệt không",
       ["người lgbtq cần chăm sóc sức khỏe", "lgbtq sức khỏe", "lgbtq có cần chăm sóc đặc biệt"],
       "Người LGBTQ+ vẫn cần chăm sóc sức khỏe như mọi người, đồng thời có thể cần hỗ trợ thêm về sức khỏe tâm lý, phòng tránh STI, an toàn trong quan hệ và đối phó kỳ thị.",
       "Hãy tìm dịch vụ y tế/tư vấn tôn trọng, không phán xét.",
       "Nên tìm hỗ trợ nếu bị kỳ thị, bạo lực, ép buộc hoặc căng thẳng kéo dài."),

    kb("lgbtq", "tôn trọng sự khác biệt giới tính",
       ["tôn trọng sự khác biệt giới tính", "tôn trọng lgbt", "ứng xử với lgbt"],
       "Tôn trọng sự khác biệt giới tính là không chế giễu, không ép thay đổi, dùng cách xưng hô phù hợp và lắng nghe người khác.",
       "Hãy tránh lan truyền định kiến hoặc thông tin sai.",
       "Nên tìm người lớn/nhà trường hỗ trợ nếu có bắt nạt hoặc kỳ thị."),

    kb("lgbtq", "bị kỳ thị giới tính nên làm gì",
       ["bị kỳ thị giới tính", "bị kỳ thị lgbt", "bị trêu vì giới tính"],
       "Bị kỳ thị có thể gây tổn thương tâm lý, nhưng đó không phải lỗi của bạn.",
       "Hãy tìm người đáng tin cậy, lưu lại bằng chứng nếu bị bắt nạt và tránh đối đầu một mình trong tình huống không an toàn.",
       "Nên báo với gia đình, giáo viên, nhà trường hoặc chuyên gia tư vấn nếu bị đe dọa/bắt nạt."),

    kb("lgbtq", "nơi hỗ trợ tư vấn LGBTQ",
       ["nơi hỗ trợ lgbtq", "tư vấn lgbt", "hỗ trợ lgbtq"],
       "Có thể tìm hỗ trợ từ người lớn đáng tin cậy, chuyên gia tâm lý, nhà trường hoặc các tổ chức cộng đồng uy tín.",
       "Ưu tiên nơi tư vấn bảo mật, tôn trọng và không phán xét.",
       "Nếu bị đe dọa hoặc bạo lực, cần tìm hỗ trợ trực tiếp từ người lớn/cơ quan phù hợp."),

    kb("consent_safety", "đồng thuận trong mối quan hệ",
       ["đồng thuận", "đồng thuận trong mối quan hệ", "consent"],
       "Đồng thuận là khi cả hai tự nguyện, tỉnh táo, hiểu rõ và có quyền dừng lại bất cứ lúc nào. Im lặng hoặc bị ép không phải là đồng thuận.",
       "Hãy tôn trọng ranh giới của bản thân và người khác.",
       "Cần tìm hỗ trợ nếu có ép buộc, đe dọa hoặc xâm hại."),

    kb("consent_safety", "nói không khi không thoải mái",
       ["nói không", "không thoải mái", "từ chối"],
       "Bạn có quyền nói không với bất kỳ điều gì khiến mình không thoải mái.",
       "Có thể nói rõ: mình không muốn, mình chưa sẵn sàng, hãy dừng lại. Tránh ở riêng nếu thấy không an toàn.",
       "Tìm người đáng tin cậy hỗ trợ nếu người kia tiếp tục ép buộc."),

    kb("consent_safety", "bị ép buộc làm điều không muốn",
       ["bị ép buộc", "bị ép làm điều không muốn", "không muốn mà bị ép"],
       "Nếu bạn bị ép làm điều mình không muốn, đó là dấu hiệu không an toàn. Bạn không có lỗi.",
       "Rời khỏi tình huống nếu có thể, lưu bằng chứng nếu bị đe dọa và tìm người lớn đáng tin cậy.",
       "Cần tìm hỗ trợ ngay nếu có đe dọa, xâm hại hoặc nguy cơ bị hại."),

    kb("consent_safety", "bảo vệ bản thân trên mạng xã hội",
       ["bảo vệ bản thân trên mạng", "an toàn mạng xã hội", "bảo mật trên mạng"],
       "Trên mạng xã hội, cần cẩn trọng với người lạ, thông tin cá nhân, hình ảnh riêng tư và lời rủ rê gặp mặt.",
       "Không gửi thông tin/hình ảnh nhạy cảm, bật bảo mật tài khoản và chặn/báo cáo người quấy rối.",
       "Tìm người lớn hỗ trợ nếu bị đe dọa, tống tiền hoặc phát tán thông tin riêng tư."),

    kb("consent_safety", "khi bị quấy rối nên tìm ai giúp",
       ["bị quấy rối", "ai giúp khi bị quấy rối", "quấy rối nên làm gì"],
       "Khi bị quấy rối, điều quan trọng là bảo vệ an toàn của bạn trước, không tự chịu đựng một mình.",
       "Nói với người lớn đáng tin cậy, giáo viên, nhà trường hoặc cơ quan hỗ trợ. Lưu lại bằng chứng nếu an toàn.",
       "Cần tìm hỗ trợ ngay nếu có đe dọa, bám theo, ép buộc hoặc xâm hại."),

    kb("consent_safety", "chia sẻ hình ảnh riêng tư nguy hiểm",
       ["chia sẻ hình ảnh riêng tư", "gửi ảnh nhạy cảm", "ảnh riêng tư"],
       "Chia sẻ hình ảnh riêng tư có thể gây rủi ro bị lưu lại, phát tán, đe dọa hoặc ép buộc.",
       "Không gửi nếu bạn không hoàn toàn an toàn và tự nguyện. Nếu bị ép, hãy từ chối và tìm người hỗ trợ.",
       "Cần báo người lớn/cơ quan hỗ trợ nếu bị đe dọa phát tán ảnh."),

    kb("consent_safety", "nhận biết hành vi không an toàn",
       ["hành vi không an toàn", "dấu hiệu không an toàn", "mối quan hệ độc hại"],
       "Hành vi không an toàn gồm ép buộc, đe dọa, kiểm soát, xúc phạm, ép gửi ảnh, ép gặp riêng hoặc không tôn trọng lời từ chối.",
       "Tin vào cảm giác không thoải mái của mình và đặt ranh giới rõ ràng.",
       "Tìm hỗ trợ nếu bị kiểm soát, đe dọa hoặc sợ hãi."),

    kb("consent_safety", "gặp người quen qua mạng một mình",
       ["gặp người quen qua mạng", "gặp người lạ trên mạng", "gặp một mình"],
       "Gặp người quen qua mạng một mình có thể không an toàn, đặc biệt nếu bạn chưa biết rõ họ ngoài đời.",
       "Không nên đi một mình; hãy báo người thân, chọn nơi công cộng và ưu tiên an toàn cá nhân.",
       "Hủy cuộc gặp và tìm hỗ trợ nếu người đó ép giữ bí mật, rủ đến nơi riêng tư hoặc làm bạn sợ."),

    kb("hygiene_body", "bao lâu vệ sinh vùng kín một lần",
       ["bao lâu vệ sinh vùng kín", "vệ sinh vùng kín mấy lần", "rửa vùng kín bao lâu"],
       "Thường nên vệ sinh vùng kín hằng ngày và sau khi ra nhiều mồ hôi, nhưng cần nhẹ nhàng, không rửa quá mức.",
       "Rửa bên ngoài bằng nước sạch hoặc sản phẩm dịu nhẹ nếu phù hợp, lau khô và thay đồ lót sạch.",
       "Nên đi khám nếu có ngứa, mùi hôi hoặc dịch bất thường."),

    kb("hygiene_body", "dùng xà phòng mạnh vệ sinh vùng kín",
       ["xà phòng mạnh", "rửa vùng kín bằng xà phòng", "xà phòng vệ sinh vùng kín"],
       "Xà phòng mạnh hoặc sản phẩm có mùi thơm mạnh có thể gây kích ứng vùng nhạy cảm.",
       "Nên dùng nước sạch hoặc sản phẩm dịu nhẹ, chỉ vệ sinh bên ngoài.",
       "Ngưng dùng và đi khám nếu ngứa rát, đỏ, đau hoặc dịch bất thường."),

    kb("hygiene_body", "vệ sinh vùng kín khi có kinh",
       ["vệ sinh khi có kinh", "vệ sinh vùng kín khi có kinh", "đang có kinh vệ sinh"],
       "Khi có kinh, cần vệ sinh nhẹ nhàng và thay băng vệ sinh đều để giảm ẩm bí và mùi khó chịu.",
       "Rửa bên ngoài, lau khô, thay băng đúng lúc và không thụt rửa sâu.",
       "Nên đi khám nếu mùi hôi rõ, ngứa rát, đau nhiều hoặc ra máu bất thường."),

    kb("hygiene_body", "giảm mùi cơ thể tuổi dậy thì",
       ["giảm mùi cơ thể", "mùi cơ thể tuổi dậy thì", "hôi cơ thể"],
       "Mùi cơ thể ở tuổi dậy thì tăng do tuyến mồ hôi hoạt động mạnh hơn.",
       "Tắm rửa đều, thay đồ sau khi ra mồ hôi, mặc đồ thoáng và giữ nách/vùng kín khô sạch.",
       "Nên khám nếu mùi rất nặng kèm viêm da, ngứa hoặc tiết dịch bất thường."),

    kb("hygiene_body", "sản phẩm tạo mùi vùng kín",
       ["sản phẩm tạo mùi vùng kín", "xịt thơm vùng kín", "làm thơm vùng kín"],
       "Sản phẩm tạo mùi cho vùng kín có thể gây kích ứng hoặc che giấu dấu hiệu viêm nhiễm.",
       "Không nên lạm dụng sản phẩm tạo mùi. Vệ sinh nhẹ nhàng và giữ khô thoáng là quan trọng hơn.",
       "Nên đi khám nếu có mùi hôi bất thường kéo dài."),

    kb("hygiene_body", "mặc đồ lót phù hợp",
       ["đồ lót phù hợp", "mặc đồ lót như thế nào", "quần lót phù hợp"],
       "Đồ lót phù hợp nên sạch, thoáng, vừa vặn, không quá chật và được thay đều.",
       "Thay đồ lót sau khi ra nhiều mồ hôi, giặt sạch và phơi khô.",
       "Nên đi khám nếu thường xuyên ngứa, nổi mẩn hoặc khí hư bất thường."),

    kb("hygiene_body", "vệ sinh sau khi chơi thể thao",
       ["sau khi chơi thể thao vệ sinh", "ra mồ hôi vùng kín", "tập thể thao vệ sinh"],
       "Sau khi chơi thể thao, mồ hôi và quần áo ẩm có thể làm vùng kín bí và dễ kích ứng.",
       "Nên thay đồ ướt, tắm rửa nhẹ nhàng, lau khô và mặc đồ thoáng.",
       "Nên đi khám nếu ngứa, mùi hôi hoặc dịch bất thường kéo dài."),

    kb("hygiene_body", "vệ sinh không đúng gây viêm nhiễm",
       ["vệ sinh không đúng", "vệ sinh sai gây viêm", "vì sao vệ sinh sai dễ viêm"],
       "Vệ sinh không đúng như thụt rửa sâu, dùng xà phòng mạnh, mặc đồ ẩm/chật có thể làm mất cân bằng và gây kích ứng hoặc viêm nhiễm.",
       "Rửa nhẹ bên ngoài, giữ khô thoáng, thay đồ sạch và tránh sản phẩm mạnh.",
       "Nên đi khám nếu triệu chứng không cải thiện hoặc có dấu hiệu viêm nhiễm.")
]

df = pd.DataFrame(knowledge_base)
documents = (
    df["intent"].astype(str) + " "
    + df["topic"].astype(str) + " "
    + df["keywords"].apply(lambda x: " ".join(x)) + " "
    + df["answer"].astype(str) + " "
    + df["advice"].astype(str)
).tolist()

vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 3),
    max_features=25000,
    sublinear_tf=True
)
tfidf_matrix = vectorizer.fit_transform(documents)

NORMALIZATION_MAP = {
    "ngày dâu": "kinh nguyệt",
    "đèn đỏ": "kinh nguyệt",
    "tới tháng": "kinh nguyệt",
    "đến tháng": "kinh nguyệt",
    "bị tháng": "kinh nguyệt",
    "chưa tới tháng": "trễ kinh",
    "rụng dâu": "kinh nguyệt",
    "ra máu nhiều": "rong kinh",
    "ra dịch": "khí hư",
    "dịch có mùi": "khí hư mùi hôi",
    "đi tiểu rát": "tiểu buốt",
    "đi tiểu đau": "tiểu buốt",
    "nổi hột": "mụn",
    "khó chịu dưới đó": "vùng kín khó chịu",
    "ra nước vàng": "dịch vàng",
    "hôi": "mùi hôi",
    "dính bầu": "mang thai",
    "quay tay": "thủ dâm",
    "tự sướng": "thủ dâm",
    "không muốn mà bắt": "ép buộc",
}

INTENT_KEYWORDS = {
    "menstruation": [
        "kinh nguyệt", "kinh không đều", "kinh nguyệt không đều",
        "rối loạn kinh nguyệt", "trễ kinh", "chậm kinh", "mất kinh",
        "rong kinh", "đau bụng kinh", "chu kỳ kinh", "máu kinh",
        "kinh tới sớm", "kinh tới muộn", "kinh đến sớm", "kinh đến muộn"
    ],
    "pregnancy_risk": [
        "có thai", "mang thai", "dính bầu", "cọ xát", "xuất tinh",
        "trễ kinh sau quan hệ", "que thử", "chậm kinh sau quan hệ",
        "quan hệ lần đầu"
    ],
    "contraception": [
        "bao cao su", "thuốc tránh thai", "rách bao", "tuột bao",
        "tránh thai khẩn cấp", "quên uống thuốc", "dùng bao"
    ],
    "gynecology": [
        "khí hư", "dịch âm đạo", "dịch trắng", "dịch vàng", "dịch xanh",
        "vón cục", "mùi hôi", "viêm âm đạo", "ngứa vùng kín"
    ],
    "urinary": [
        "tiểu buốt", "tiểu rắt", "đau khi đi tiểu", "viêm đường tiểu", "tiểu ra máu"
    ],
    "sti_std": [
        "ngứa", "rát", "mụn", "vết loét", "hiv", "hpv", "lậu",
        "giang mai", "herpes", "sùi mào gà", "sti", "std", "bệnh lây",
        "quan hệ không an toàn"
    ],
    "puberty": [
        "dậy thì", "vỡ giọng", "mộng tinh", "mọc lông", "chưa có kinh",
        "chưa vỡ giọng", "thủ dâm", "tự sướng", "mùi cơ thể"
    ],
    "hygiene_body": [
        "vệ sinh", "dung dịch vệ sinh", "thụt rửa", "vùng kín",
        "xà phòng", "đồ lót", "chơi thể thao"
    ],
    "lgbtq": [
        "gay", "lesbian", "bisexual", "lgbt", "lgbtq", "đồng tính",
        "song tính", "come out", "chuyển giới", "xu hướng tính dục"
    ],
    "consent_safety": [
        "ép", "ép buộc", "không muốn", "gửi ảnh", "ảnh nhạy cảm",
        "xâm hại", "lạm dụng", "đồng thuận", "đe dọa", "quấy rối",
        "người quen qua mạng"
    ],
    "sexual_function": [
        "xuất tinh sớm", "rối loạn cương", "khô rát", "đau khi quan hệ", "ham muốn"
    ]
}

DOMAIN_KEYWORDS = sorted(set(sum(INTENT_KEYWORDS.values(), [])))

OUT_OF_SCOPE_KEYWORDS = [
    "thời tiết", "điện thoại", "laptop", "game", "nấu", "cháo", "bóng đá",
    "toán", "code", "python", "palworld", "liên quân", "giảm cân", "visa", "ngân hàng"
]

class MemoryAgent:
    def __init__(self):
        self.state = {"last_intent": None, "context": {}, "asked": set()}

    def update(self, intent=None, context=None):
        if intent:
            self.state["last_intent"] = intent
        if context:
            for k, v in context.items():
                if v not in [None, False, [], ""]:
                    self.state["context"][k] = v

    def merge_context(self, context):
        merged = dict(self.state["context"])
        for k, v in context.items():
            if v not in [None, False, [], ""]:
                merged[k] = v
        return merged

    def reset(self):
        self.state = {"last_intent": None, "context": {}, "asked": set()}

    def already_asked(self, key):
        if key in self.state["asked"]:
            return True
        self.state["asked"].add(key)
        return False

memory_agent = MemoryAgent()

class InputAnalysisAgent:
    def normalize(self, text):
        text = (text or "").lower().strip()
        for old, new in NORMALIZATION_MAP.items():
            text = text.replace(old, new)
        return text

    def run(self, message):
        text = self.normalize(message)

        if text in ["chào", "xin chào", "hello", "hi", "bot ơi", "bạn ơi", "alo"]:
            return {"text": text, "intent": "greeting"}

        if any(x in text for x in ["cảm ơn", "tạm biệt", "bye", "kết thúc", "xong rồi"]):
            return {"text": text, "intent": "ending"}

        if any(x in text for x in OUT_OF_SCOPE_KEYWORDS) and not any(x in text for x in DOMAIN_KEYWORDS):
            return {"text": text, "intent": "out_of_scope"}

        best_intent = None
        best_hit = 0

        for item in knowledge_base:
            hit = 0
            for kw in item["keywords"]:
                if kw in text:
                    hit += len(kw.split())
            if hit > best_hit:
                best_hit = hit
                best_intent = item["intent"]

        if best_intent and best_hit >= 2:
            return {"text": text, "intent": best_intent}

        scores = {}
        for intent, kws in INTENT_KEYWORDS.items():
            scores[intent] = sum(1 for kw in kws if kw in text)

        best_intent = max(scores, key=scores.get)

        if scores[best_intent] == 0:
            if len(text.split()) <= 5:
                return {"text": text, "intent": "missing_info"}
            return {"text": text, "intent": "out_of_scope"}

        return {"text": text, "intent": best_intent}

class ContextExtractionAgent:
    def run(self, text):
        context = {
            "age": None,
            "gender": None,
            "symptoms": [],
            "duration": None,
            "has_sexual_risk": False,
            "used_condom": None,
            "period_late": False,
            "stress": False,
            "fear": False,
            "coercion": False,
            "cycle_change": False
        }

        age = re.search(r"(\d{1,2})\s*tuổi", text)
        if age:
            context["age"] = age.group(1)

        if any(x in text for x in ["nam", "con trai"]):
            context["gender"] = "nam"
        if any(x in text for x in ["nữ", "con gái"]):
            context["gender"] = "nữ"

        symptoms = [
            "ngứa", "rát", "đau", "mụn", "loét", "khí hư", "mùi hôi",
            "tiểu buốt", "tiểu rắt", "trễ kinh", "mộng tinh", "vỡ giọng",
            "khô rát", "rong kinh", "đau bụng kinh", "mất kinh",
            "dịch vàng", "dịch xanh", "vón cục", "kinh không đều"
        ]
        context["symptoms"] = [s for s in symptoms if s in text]

        duration = re.search(r"(\d+)\s*(ngày|tuần|tháng|năm)", text)
        if duration:
            context["duration"] = duration.group(0)

        if any(x in text for x in ["quan hệ", "cọ xát", "xuất tinh", "không dùng bao", "rách bao", "tuột bao"]):
            context["has_sexual_risk"] = True

        if "không dùng bao" in text:
            context["used_condom"] = False
        elif "bao cao su" in text or "dùng bao" in text:
            context["used_condom"] = True

        if any(x in text for x in ["trễ kinh", "chậm kinh", "mất kinh"]):
            context["period_late"] = True

        if any(x in text for x in ["stress", "căng thẳng", "thức khuya", "áp lực", "lo lắng"]):
            context["stress"] = True

        if any(x in text for x in ["lo quá", "sợ", "hoang mang", "ngại", "xấu hổ"]):
            context["fear"] = True

        if any(x in text for x in ["ép", "ép buộc", "không muốn", "đe dọa", "xâm hại", "lạm dụng", "quấy rối"]):
            context["coercion"] = True

        if any(x in text for x in ["kinh đến sớm", "kinh đến muộn", "kinh tới sớm", "kinh tới muộn", "kinh không đều", "chu kỳ thay đổi"]):
            context["cycle_change"] = True

        return context

class SafetyTriageAgent:
    def run(self, intent, context, text):
        red_flags = []

        if context.get("coercion"):
            red_flags.append("Có dấu hiệu ép buộc hoặc không an toàn.")

        if any(x in text for x in ["chảy máu nhiều", "đau dữ dội", "ngất", "sốt cao", "mệt lả", "tiểu ra máu"]):
            red_flags.append("Có dấu hiệu cần được hỗ trợ y tế sớm.")

        if red_flags:
            return {"level": "🔴 CẦN HỖ TRỢ SỚM", "red_flags": red_flags}

        if intent in ["sti_std", "gynecology", "urinary", "consent_safety"]:
            return {"level": "🟠 CẦN THEO DÕI/KHÁM NẾU KÉO DÀI", "red_flags": []}

        if intent in ["pregnancy_risk", "contraception", "menstruation"]:
            return {"level": "🟡 CẦN THEO DÕI THÊM", "red_flags": []}

        return {"level": "🟢 CHƯA THẤY DẤU HIỆU KHẨN CẤP", "red_flags": []}

class MissingInfoAgent:
    def run(self, intent, context, text):
        questions = []

        if intent == "pregnancy_risk":
            if not context.get("has_sexual_risk"):
                questions.append("Tình huống là cọ xát ngoài, có quan hệ hay có xuất tinh gần vùng kín không?")
            if not context.get("duration"):
                questions.append("Việc đó xảy ra cách đây bao lâu?")
            if not context.get("period_late"):
                questions.append("Hiện có trễ kinh hoặc gần đến ngày kinh chưa?")

        elif intent in ["sti_std", "gynecology", "urinary"]:
            if not context.get("symptoms"):
                questions.append("Triệu chứng cụ thể là gì: ngứa, rát, đau, nổi mụn, dịch lạ hay tiểu buốt?")
            if not context.get("duration"):
                questions.append("Triệu chứng xuất hiện bao lâu rồi?")
            if intent == "sti_std" and not context.get("has_sexual_risk"):
                questions.append("Gần đây có quan hệ nguy cơ hoặc không dùng bao không?")

        elif intent == "puberty":
            if not context.get("age"):
                questions.append("Bạn bao nhiêu tuổi?")
            if not context.get("gender"):
                questions.append("Bạn là nam hay nữ?")
            if not context.get("symptoms"):
                questions.append("Biểu hiện cụ thể bạn đang lo là gì?")

        questions = questions[:2]

        if questions:
            key = intent + "|" + "|".join(questions)
            if not memory_agent.already_asked(key):
                return "Mình cần thêm một chút thông tin để tư vấn đúng hơn:\n\n" + "\n".join(f"- {q}" for q in questions)

        return None

class KnowledgeRetrievalAgent:
    def run(self, text, intent):
        candidate_indexes = df[df["intent"] == intent].index.tolist()

        best_idx = None
        best_hit = 0

        for i in candidate_indexes:
            item = df.iloc[i]
            hit = 0
            for kw in item["keywords"]:
                if kw in text:
                    hit += len(kw.split())
            if hit > best_hit:
                best_hit = hit
                best_idx = i

        if best_idx is not None and best_hit >= 2:
            row = df.iloc[best_idx].to_dict()
            row["score"] = 1.0
            return row

        query = f"{intent} {text}"
        user_vec = vectorizer.transform([query])
        scores = cosine_similarity(user_vec, tfidf_matrix).flatten()

        if candidate_indexes:
            best_idx = max(candidate_indexes, key=lambda i: scores[i])
        else:
            best_idx = int(scores.argmax())

        best_score = float(scores[best_idx])
        threshold = 0.13 if intent in ["menstruation", "pregnancy_risk", "puberty", "hygiene_body"] else 0.16

        if best_score < threshold:
            return None

        row = df.iloc[best_idx].to_dict()
        row["score"] = best_score
        return row

class AdvicePlanningAgent:
    def run(self, intent, context, knowledge, safety, text):
        if knowledge is None:
            return None

        advice = knowledge["advice"]

        if intent == "menstruation":
            if context.get("cycle_change"):
                advice += "\n- Chu kỳ đến sớm hoặc muộn đôi khi chỉ là dao động tạm thời, nhất là khi stress hoặc thức khuya."
            if "đau bụng kinh" in context.get("symptoms", []):
                advice += "\n- Có thể chườm ấm bụng dưới và nghỉ ngơi; nếu đau dữ dội thì không nên cố chịu."
            if "rong kinh" in context.get("symptoms", []):
                advice += "\n- Hãy theo dõi lượng máu, số ngày ra máu và dấu hiệu chóng mặt/mệt lả."
            if context.get("stress"):
                advice += "\n- Stress và thức khuya có thể làm chu kỳ kinh bị lệch."

        if intent in ["gynecology", "sti_std"]:
            advice += "\n- Không thụt rửa sâu, không tự đặt thuốc hoặc bôi thuốc mạnh khi chưa rõ nguyên nhân."

        if intent == "urinary":
            advice += "\n- Uống đủ nước, không nhịn tiểu và không tự dùng kháng sinh."

        if intent == "consent_safety":
            advice += "\n- Nếu có tin nhắn đe dọa hoặc ép buộc, hãy lưu lại bằng chứng và tìm người đáng tin cậy hỗ trợ."

        return advice

class ResponseGenerationAgent:
    def empathy(self, context):
        if context.get("coercion"):
            return "Trước hết, bạn không có lỗi nếu đang bị ép làm điều mình không muốn."
        if context.get("fear"):
            return "Mình hiểu là bạn đang lo lắng, mình sẽ giải thích theo hướng dễ hiểu và an toàn nhé."
        return "Mình sẽ dựa trên thông tin bạn đưa để nhận định ban đầu nhé."

    def run(self, intent, context, safety, knowledge, advice):
        if knowledge is None:
            return (
                "Mình chưa tìm thấy tri thức đủ khớp trong cơ sở dữ liệu hiện tại nên chưa nên kết luận.\n\n"
                "Bạn có thể mô tả rõ hơn tình huống, thời gian xảy ra và triệu chứng chính không?"
            )

        red_flag_text = ""
        if safety["red_flags"]:
            red_flag_text = "\n".join(f"- {x}" for x in safety["red_flags"]) + "\n\n"

        return f"""
{self.empathy(context)}

Nhận định ban đầu:
{knowledge["answer"]}

Vì sao có thể như vậy:
Câu hỏi của bạn thuộc nhóm "{intent}". Hệ thống chỉ đưa ra nhận định ban đầu dựa trên thông tin bạn cung cấp, không chẩn đoán chắc chắn.

Mức độ cần lưu ý:
{safety["level"]}

{red_flag_text}Bạn nên làm gì:
{advice}

Khi nào cần đi khám/tìm hỗ trợ:
{knowledge["seek_help"]}

Lưu ý an toàn:
Thông tin này chỉ mang tính tham khảo, không thay thế tư vấn trực tiếp từ bác sĩ, dược sĩ hoặc chuyên gia phù hợp.
""".strip()

input_agent = InputAnalysisAgent()
context_agent = ContextExtractionAgent()
safety_agent = SafetyTriageAgent()
missing_agent = MissingInfoAgent()
retrieval_agent = KnowledgeRetrievalAgent()
advice_agent = AdvicePlanningAgent()
response_agent = ResponseGenerationAgent()

def chatbot(message, history):
    if not message or not message.strip():
        return "Bạn hãy nhập câu hỏi để mình hỗ trợ nhé."

    analysis = input_agent.run(message)
    text = analysis["text"]
    intent = analysis["intent"]

    if intent == "greeting":
        return (
            "Xin chào 👋\n\n"
            "Mình là trợ lý ảo AI hỗ trợ trả lời các câu hỏi thường gặp về sức khỏe giới tính.\n"
            "Bạn có thể mô tả tình huống của mình, mình sẽ trả lời theo hướng dễ hiểu, an toàn và không phán xét."
        )

    if intent == "ending":
        memory_agent.reset()
        return "Cuộc trò chuyện đã được kết thúc. Mình đã làm mới ngữ cảnh cho câu hỏi tiếp theo."

    if intent == "out_of_scope":
        return (
            "Mình chỉ hỗ trợ các câu hỏi về sức khỏe giới tính, sức khỏe sinh sản, dậy thì, tránh thai, "
            "bệnh lây truyền qua đường tình dục, kinh nguyệt, vệ sinh vùng kín, đồng thuận và các vấn đề liên quan.\n\n"
            "Bạn hãy hỏi lại đúng phạm vi để mình hỗ trợ chính xác hơn nhé."
        )

    if intent == "missing_info":
        return (
            "Mình chưa đủ thông tin để nhận định.\n\n"
            "Bạn có thể nói rõ hơn: vấn đề đang gặp là gì, xảy ra bao lâu rồi, có triệu chứng nào kèm theo và điều bạn lo nhất là gì?"
        )

    context = context_agent.run(text)
    context = memory_agent.merge_context(context)
    memory_agent.update(intent=intent, context=context)

    safety = safety_agent.run(intent, context, text)

    followup = missing_agent.run(intent, context, text)
    if followup:
        return followup

    knowledge = retrieval_agent.run(text, intent)
    advice = advice_agent.run(intent, context, knowledge, safety, text)

    return response_agent.run(intent, context, safety, knowledge, advice)

custom_css = """
.gradio-container {
    background: #0f172a !important;
    color: white !important;
}

#main-title {
    text-align: center;
    padding: 18px;
    border-radius: 18px;
    background: linear-gradient(135deg, #1e293b, #334155);
    margin-bottom: 18px;
}

#side-panel {
    background: #111827;
    border: 1px solid #334155;
    border-radius: 18px;
    padding: 16px;
}

textarea {
    border-radius: 16px !important;
}

footer {
    display: none !important;
}
"""

with gr.Blocks(css=custom_css) as demo:
    gr.Markdown(
        f"""
<div id="main-title">

# {APP_NAME}

Hỗ trợ trả lời các câu hỏi thường gặp về sức khỏe giới tính theo hướng dễ hiểu, an toàn và không phán xét.

</div>
""",
    )

    with gr.Row():
        with gr.Column(scale=3):
            gr.ChatInterface(
                fn=chatbot,
                examples=[],
                textbox=gr.Textbox(
                    placeholder="Nhập câu hỏi của bạn tại đây...",
                    container=True,
                    scale=7
                )
            )

        with gr.Column(scale=1):
            gr.Markdown(
                """
<div id="side-panel">

### Phạm vi hỗ trợ

- Dậy thì nam/nữ
- Kinh nguyệt, trễ kinh
- Tránh thai, bao cao su
- Viêm nhiễm vùng kín
- STI/STD
- LGBTQ+
- Đồng thuận và an toàn cá nhân
- Vệ sinh vùng nhạy cảm

### Lưu ý

Thông tin chỉ mang tính tham khảo, không thay thế tư vấn y tế trực tiếp.

</div>
""",
            )

demo.launch(share=True)
